# EloSense: Chess Elo Prediction from Game Metadata

Can we guess a player's rating band just from how a game was played (time control, result type, format), without looking at their actual rating?

## 1. Setup & Data Loading

In [ ]:
import pandas as pd

DATA_PATH = "../data/club_games_data.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.dtypes)
df.head()

## 2. Data Quality Checks

In [ ]:
print(df.isnull().sum())
print(df.duplicated().sum())

# Check whether each game is duplicated as a mirrored white/black row
print(df["pgn"].duplicated().sum())

### Leakage decision

Goal: predict a rating band (not the exact rating), using only game format and the opponent's rating band. Keeps this a classification task and avoids the trivial version of the problem.

Columns dropped from features, and why:

- `white_username`, `black_username`, `white_id`, `black_id`: these identify the player. A model could just memorize a specific person's rating instead of learning from the game itself.
- `pgn`, `fen`: move-level data, saving this for a later version.
- Opponent's exact rating: only using their rating band, so the task isn't trivial.
- The target player's own past rating isn't even in this dataset per row, so nothing to drop there.

Columns kept as features: `time_class`, `time_control`, `rules`, `rated`, `white_result`/`black_result`, and opponent rating band.

## 3. EDA: Rating Distributions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["white_rating"], bins=50)
axes[0].set_title("White rating distribution")
axes[0].set_xlabel("Rating")

axes[1].hist(df["black_rating"], bins=50)
axes[1].set_title("Black rating distribution")
axes[1].set_xlabel("Rating")

plt.tight_layout()
plt.show()

## 4. EDA: Time Class and Result Breakdowns

In [ ]:
print(df["time_class"].value_counts())
print()
print(df["white_result"].value_counts())

bins = [0, 1000, 1400, 1800, 5000]
labels = ["<1000", "1000-1400", "1400-1800", "1800+"]
df["white_band"] = pd.cut(df["white_rating"], bins=bins, labels=labels)

top_results = ["checkmated", "resigned", "timeout", "win", "agreed"]
sub = df[df["white_result"].isin(top_results)]
pd.crosstab(sub["white_band"], sub["white_result"], normalize="index").round(3)

### Findings

- Blitz is by far the most common format (~29k games), followed by bullet (~22.5k), rapid (~13k), and daily is rare (~2k).
- Win rate for white climbs steadily with rating band: 49.3% under 1000, up to 57.5% at 1800+.
- Getting checkmated drops as rating rises: 15.3% under 1000 vs 10.7% at 1800+. Stronger players avoid getting mated more often.
- Losing on time also drops with rating: 17.9% under 1000 vs 13.8% at 1800+, suggesting better time management at higher levels.
- Resigning stays roughly flat across bands (~17-19%), so how often someone resigns isn't a strong skill signal on its own.

## 5. Rating Band Bucketing

In [ ]:
RATING_BINS = [0, 1000, 1400, 1800, 5000]
RATING_LABELS = ["<1000", "1000-1400", "1400-1800", "1800+"]

def rating_to_band(rating):
    return pd.cut([rating], bins=RATING_BINS, labels=RATING_LABELS)[0]

df["white_band"] = pd.cut(df["white_rating"], bins=RATING_BINS, labels=RATING_LABELS)
df["black_band"] = pd.cut(df["black_rating"], bins=RATING_BINS, labels=RATING_LABELS)

df[["white_rating", "white_band", "black_rating", "black_band"]].head()

## 6. Feature Encoding

In [ ]:
def result_to_outcome(result):
    if result == "win":
        return "win"
    if result in ["agreed", "repetition", "stalemate", "insufficient", "timevsinsufficient", "50move"]:
        return "draw"
    return "loss"

def result_to_reason(result):
    if result in ["checkmated", "resigned", "timeout", "abandoned"]:
        return result
    if result == "win":
        return "win"
    return "draw"

df["result_outcome"] = df["white_result"].apply(result_to_outcome)
df["result_reason"] = df["white_result"].apply(result_to_reason)

df[["white_result", "result_outcome", "result_reason"]].head()

In [ ]:
# Feature columns:
# - time_class, rules: game format
# - rated: whether the game affected Elo
# - result_outcome, result_reason: how the game ended (derived above)
# - black_band: opponent's rating band, not their exact rating (see leakage decision)
# Target: white_band

feature_cols = ["time_class", "rules", "rated", "result_outcome", "result_reason", "black_band"]

X = pd.get_dummies(df[feature_cols], columns=["time_class", "rules", "result_outcome", "result_reason", "black_band"])
y = df["white_band"]

print(X.shape, y.shape)
X.head()

## 7. Leakage Ablation Test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

feature_cols_with = ["time_class", "rules", "rated", "result_outcome", "result_reason", "black_band"]
feature_cols_without = ["time_class", "rules", "rated", "result_outcome", "result_reason"]

ablation_results = {}
for name, cols in [("with black_band", feature_cols_with), ("without black_band", feature_cols_without)]:
    cat_cols = [c for c in cols if c != "rated"]
    X_ablate = pd.get_dummies(df[cols], columns=cat_cols)
    Xtr, Xte, ytr, yte = train_test_split(X_ablate, y, test_size=0.2, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(Xtr, ytr)
    acc = accuracy_score(yte, clf.predict(Xte))
    ablation_results[name] = acc

ablation_results

Accuracy with `black_band`: **85.2%**. Without it: **39.1%**. A 46-point drop.

Opponent's rating band is carrying most of the signal here, which makes sense: chess.com's matchmaking pairs players close in rating, so knowing the opponent's band is close to knowing the answer. The other features (time_class, rules, rated, result type) add real but much smaller signal on their own. Worth keeping this in mind when reading later model results, since most of the accuracy is coming from matchmaking, not from "how the game was played."

## 8. Baseline Model

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

baseline = LogisticRegression(max_iter=1000)
baseline.fit(Xtr, ytr)
baseline_preds = baseline.predict(Xte)

print("accuracy:", accuracy_score(yte, baseline_preds))
print("macro f1:", f1_score(yte, baseline_preds, average="macro"))
print()
print(classification_report(yte, baseline_preds, labels=RATING_LABELS))

baseline_cm = confusion_matrix(yte, baseline_preds, labels=RATING_LABELS)
baseline_cm

Logistic regression baseline: **85.2% accuracy**, **84.2% macro F1**. Performs worst on the 1800+ band (0.79 F1, smallest class) and best on <1000 (0.88 F1). Confusion is mostly between adjacent bands (e.g. 1000-1400 vs 1400-1800), not distant ones, which is the more forgivable kind of error.

## 9. Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Quick sweep over a few basic hyperparameter combos
for n_estimators, max_depth in [(100, None), (200, 12), (300, 16)]:
    rf_trial = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
    rf_trial.fit(Xtr, ytr)
    trial_preds = rf_trial.predict(Xte)
    print(n_estimators, max_depth, accuracy_score(yte, trial_preds), f1_score(yte, trial_preds, average="macro"))

All three combos land around 86% accuracy, so going with `n_estimators=200, max_depth=12` (regularized depth, no meaningful accuracy tradeoff).

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)
rf_preds = rf.predict(Xte)

print("accuracy:", accuracy_score(yte, rf_preds))
print("macro f1:", f1_score(yte, rf_preds, average="macro"))
print()
print(classification_report(yte, rf_preds, labels=RATING_LABELS))

rf_cm = confusion_matrix(yte, rf_preds, labels=RATING_LABELS)
rf_cm

## 10. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "model": ["Logistic Regression (baseline)", "Random Forest"],
    "accuracy": [accuracy_score(yte, baseline_preds), accuracy_score(yte, rf_preds)],
    "macro_f1": [f1_score(yte, baseline_preds, average="macro"), f1_score(yte, rf_preds, average="macro")],
})
comparison

Random forest edges out logistic regression, but only by about 0.8 points of accuracy and 0.8 points of macro F1. The gap between the two models is small compared to the 46-point gap from the ablation test, so the choice of model matters a lot less here than the choice of features.

## 11. Final Evaluation Plots

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay(confusion_matrix=rf_cm, display_labels=RATING_LABELS)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Random Forest: Confusion Matrix")
plt.tight_layout()
plt.savefig("../images/confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(7, 6))
importances.iloc[::-1].plot(kind="barh", ax=ax)
ax.set_title("Random Forest: Top 15 Feature Importances")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("../images/feature_importance.png", dpi=150)
plt.show()

The four `black_band` dummy features dominate importance (roughly 93% of it combined), matching the ablation test. `rated` and `time_class` show up next but are far smaller contributors.